# 02. 출시 전 체크리스트 생성을 위한 전체 게임 LLM 리뷰 분석

`01_preprocess_reviews.ipynb`전처리 파일에서 만든 후보 데이터를 받아서\
분석 조건 필터링과 샘플링을 적용한 뒤 **PydanticAI + Vertex AI 기반 Gemini**로 Steam 리뷰 감성·이슈 분석을 실행한다.


## 역할
1. 전처리 완료 후보 리뷰를 불러온다.
2. 장르, 출시일, 언어, 출시 구간, 리뷰 신뢰도 조건을 적용한다.
3. 게임별 리뷰 수를 제한하고 Steam 긍정/부정 라벨을 샘플링한다.
4. LLM 입력 파일을 저장한다.
5. Vertex AI 기반 Gemini 모델로 리뷰별 감정과 이슈를 분석한다.
6. 보고서에서 사용할 CSV 산출물을 저장한다.

## 이 코드에서 사용하는 주요 입력
- `llm_preprocessed_reviews.csv`
- `llm_candidate_game_summary.csv`

## 이 코드에서 생성하는 주요 산출물
- `llm_input_reviews.csv`
- `llm_review_analysis_result.csv`
- `llm_issue_tags_flat.csv`

## 분석 흐름

```text
리뷰 데이터가 있는 게임 전체
        ↓
출시 초기 D0-D30 리뷰 필터링
        ↓
영어 리뷰 / Steam 구매 / 무료 수령 제외 / 얼리액세스 제외
        ↓
게임별 최대 N개 리뷰 샘플링
부정 리뷰를 우선 확보하되, 부족한 경우 남은 긍정 리뷰로 채움
        ↓
LLM 입력 파일 저장
llm_input_reviews.csv
        ↓
Vertex AI Gemini 호출
PydanticAI 구조화 출력
        ↓
리뷰 단위 결과 저장
llm_review_analysis_result.csv
        ↓
이슈 태그 펼치기
llm_issue_tags_flat.csv
```


# 0. 환경설정

In [1]:
# ============================================================
# 기본 라이브러리
# ============================================================
import os
import ast
import json
import time
import asyncio
import platform
from pathlib import Path
from typing import List, Literal, Optional
from datetime import datetime

# ============================================================
# 데이터 분석용 라이브러리
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# ============================================================
# 환경변수 / 진행률 / LLM 출력 스키마 관련 라이브러리
# ============================================================
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tqdm.auto import tqdm
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel, GoogleModelSettings
from pydantic_ai.providers.google import GoogleProvider
from google import genai
from google.genai import types



# ============================================================
# 한글 폰트 설정
# ============================================================
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":  # macOS
    plt.rcParams["font.family"] = "AppleGothic"
else:  # Linux
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (12, 6)

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

c:\Users\joon5\Documents\github\steam-indie-game-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 경로 설정

In [2]:
# ============================================================
# 프로젝트 경로 설정
# ============================================================
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정한다.
ROOT = Path.cwd()

if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

# ============================================================
# 전처리 산출물 폴더
# ============================================================
PREPROCESS_OUTPUT_DIR = ROOT / "data" / "outputs" / "preprocess_llm"

# 전처리 파일에서 만든 후보 리뷰 데이터
PREPROCESSED_REVIEWS_PATH = PREPROCESS_OUTPUT_DIR / "llm_preprocessed_reviews.csv"

# ============================================================
# 출시 전 LLM 분석 결과 저장 폴더
# ============================================================
# Streamlit에서 사용할 최종 데이터는 master에 저장한다.
PRELAUNCH_OUTPUT_DIR = ROOT / "data" / "outputs" / "prelaunch"
OUTPUT_DIR = PRELAUNCH_OUTPUT_DIR / "master"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 하위 산출물 폴더
CHECKLIST_DATA_DIR = OUTPUT_DIR / "prelaunch_checklist_data"
TABLEAU_DIR = OUTPUT_DIR / "tableau_dashboard_csv"

CHECKLIST_DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLEAU_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# LLM 실행 직전 산출물
# ============================================================
LLM_INPUT_PATH = OUTPUT_DIR / "llm_input_reviews.csv"
FILTER_LOG_PATH = OUTPUT_DIR / "llm_input_filter_log.csv"
SAMPLE_SUMMARY_PATH = OUTPUT_DIR / "llm_input_sample_summary.csv"

# ============================================================
# LLM 분석 기본 산출물
# ============================================================
CHECKPOINT_PATH = OUTPUT_DIR / "llm_review_analysis_checkpoint.json"
RESULT_JSON_PATH = OUTPUT_DIR / "llm_review_analysis_result.json"
RESULT_CSV_PATH = OUTPUT_DIR / "llm_review_analysis_result.csv"
ISSUE_TAG_FLAT_PATH = OUTPUT_DIR / "llm_issue_tags_flat.csv"

print("ROOT:", ROOT)
print("전처리 리뷰 파일 존재:", PREPROCESSED_REVIEWS_PATH.exists())
print("결과 저장 폴더:", OUTPUT_DIR)
print("체크리스트 데이터 폴더:", CHECKLIST_DATA_DIR)
print("Tableau 데이터 폴더:", TABLEAU_DIR)

ROOT: c:\Users\joon5\Documents\github\steam-indie-game-analysis
전처리 리뷰 파일 존재: True
결과 저장 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master
체크리스트 데이터 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\prelaunch_checklist_data
Tableau 데이터 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\tableau_dashboard_csv


## Vertex AI 설정

In [3]:
# ============================================================
# Vertex AI / PydanticAI 설정
# ============================================================
# 이 셀은 실제 LLM 호출을 위한 Google Cloud 프로젝트, location, 모델명을 설정한다.

# .env 파일에 저장된 Vertex AI 설정을 현재 Python 환경으로 불러온다.
load_dotenv()
load_dotenv(ROOT / ".env", override=True)

# Google Cloud 프로젝트 ID를 읽는다.
GOOGLE_CLOUD_PROJECT = (
    os.getenv("GOOGLE_CLOUD_PROJECT")
    or os.getenv("VERTEX_PROJECT_ID")
    or os.getenv("GCP_PROJECT_ID")
)

# Vertex AI location을 읽는다.
GOOGLE_CLOUD_LOCATION = (
    os.getenv("GOOGLE_CLOUD_LOCATION")
    or os.getenv("VERTEX_LOCATION")
    or "global"
)

# 사용할 Gemini 모델명
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite")

# 프로젝트 ID가 있으면 Vertex AI Provider와 Gemini 모델 객체를 생성한다.
if GOOGLE_CLOUD_PROJECT:
    vertex_provider = GoogleProvider(
        vertexai=True,
        project=GOOGLE_CLOUD_PROJECT,
        location=GOOGLE_CLOUD_LOCATION,
    )

    vertex_model = GoogleModel(
        GEMINI_MODEL,
        provider=vertex_provider,
    )

    print("Vertex AI project:", GOOGLE_CLOUD_PROJECT)
    print("Vertex AI location:", GOOGLE_CLOUD_LOCATION)
    print("Gemini model:", GEMINI_MODEL)
    print("Vertex 모델 생성: O")
else:
    # RUN_LLM=False로 기존 결과만 읽어 후처리할 때는 프로젝트 ID가 없어도 코드을 계속 볼 수 있게 한다.
    # 단, 실제 LLM 실행 전에는 반드시 .env 또는 환경변수에 GOOGLE_CLOUD_PROJECT를 설정해야 한다.
    vertex_provider = None
    vertex_model = None

    print("Vertex AI project: 미설정")
    print("Vertex 모델 생성: X")
    print("실제 LLM 실행 전 .env에 GOOGLE_CLOUD_PROJECT를 설정하세요.")


Vertex AI project: gen-lang-client-0587784564
Vertex AI location: global
Gemini model: gemini-3.1-flash-lite
Vertex 모델 생성: O


# 1. 분석/샘플링/LLM 실행 설정
출시 전 체크리스트 생성을 위한 LLM 분석의 실행 범위, 리뷰 샘플링 방식, LLM 호출 여부를 설정한다.    
02번 코드는 여러 Steam 인디게임의 출시 초기 리뷰를 대상으로 LLM 분석을 수행하고, 이후 출시 전 체크리스트 생성에 사용할 리뷰 단위 분석 결과와 이슈 태그 단위 데이터를 만든다.

## 1-1. 실행 여부 설정

| 설정값 | 설명 |
|---|---|
| `RUN_LLM` | 실제 LLM을 호출할지 여부를 정한다. `True`이면 LLM 분석을 새로 실행하고, `False`이면 기존에 저장된 JSON/checkpoint 결과를 불러와 후처리만 진행한다. |
| `RESET_CHECKPOINT` | 기존 checkpoint를 삭제하고 처음부터 다시 실행할지 여부를 정한다. 일반적으로 최종 결과 확인 단계에서는 `False`로 둔다. |
| `RUN_CHECK_CELLS` | 필터링 결과, 샘플링 결과, 저장 결과 등 확인용 출력 셀을 실행할지 여부를 정한다. |

최종 결과물이 이미 생성되어 있는 상태에서 코드를 처음부터 다시 실행할 때는 `RUN_LLM=False`로 둔다.  
이렇게 하면 LLM을 다시 호출하지 않고 기존 분석 결과를 불러와 바로 후처리와 최종 산출물 확인 단계로 넘어갈 수 있다.

## 1-2. 분석 대상 설정

`ANALYSIS_FILTERS`는 어떤 리뷰를 LLM 분석 대상으로 사용할지 정하는 설정이다.

본 분석에서는 특정 게임 하나가 아니라, 조건을 통과한 Steam 인디게임들의 출시 초기 리뷰를 사용한다.  
출시 전 체크리스트 생성을 위해 출시 후 `D0-D30` 구간의 영어 리뷰를 대상으로 하며, Steam 구매 리뷰만 포함하고 무료 수령 리뷰와 얼리액세스 작성 리뷰는 제외한다.

| 설정값 | 설명 |
|---|---|
| `appids` | 특정 게임만 분석할 때 appid를 입력한다. 비워두면 전체 게임을 대상으로 한다. |
| `game_name_contains` | 게임명 키워드로 분석 대상을 제한할 때 사용한다. |
| `genres` | 특정 장르만 분석할 때 사용한다. 비워두면 장르 제한이 없다. |
| `categories` | 특정 Steam 카테고리만 분석할 때 사용한다. |
| `tags` | 특정 Steam 태그가 포함된 게임만 분석할 때 사용한다. |
| `release_date_from`, `release_date_to` | 분석에 포함할 게임의 출시일 범위를 정한다. |
| `languages` | 분석할 리뷰 언어를 정한다. |
| `steam_labels` | Steam 추천/비추천 리뷰 포함 기준을 정한다. |
| `release_periods` | 출시일 기준 리뷰 구간을 정한다. |
| `steam_purchase_only` | Steam 구매 리뷰만 사용할지 정한다. |
| `exclude_received_for_free` | 무료 수령 리뷰를 제외할지 정한다. |
| `exclude_early_access_reviews` | 얼리액세스 기간 작성 리뷰를 제외할지 정한다. |
| `meaningful_review_only` | 너무 짧거나 분석 의미가 낮은 리뷰를 제외할지 정한다. |

## 1-3. 샘플링 설정

LLM 분석은 비용과 시간이 발생하므로, 전체 리뷰를 모두 분석하지 않고 게임별 리뷰 수와 전체 리뷰 수에 제한을 둔다.

출시 전 체크리스트는 유사 게임에서 반복적으로 나타난 리스크를 확인하는 것이 중요하므로, Steam 비추천 리뷰를 우선 확보하는 방식으로 샘플링한다.  
다만 긍정 리뷰도 함께 포함하여 강점과 리스크를 모두 확인할 수 있도록 한다.

| 설정값 | 설명 |
|---|---|
| `TEST_N` | 빠른 테스트용으로 일부 리뷰만 사용할 때 설정한다. 최종 실행에서는 `None`으로 둔다. |
| `RANDOM_STATE` | 샘플링 결과 재현을 위한 난수 고정값이다. |
| `REVIEWS_PER_GAME` | 게임 1개에서 최대 몇 개의 리뷰를 가져올지 정한다. |
| `MAX_TOTAL_REVIEWS` | 전체 LLM 분석 리뷰 수 상한을 정한다. |
| `SAMPLE_MODE` | 리뷰 샘플링 방식을 정한다. |
| `NEGATIVE_SAMPLE_RATIO` | 게임별 샘플에서 Steam 비추천 리뷰를 우선 확보하려는 목표 비율이다. |

## 1-4. LLM 입력 및 호출 설정

LLM에는 너무 짧은 리뷰나 지나치게 긴 리뷰를 그대로 넣지 않고, 분석 가능한 길이로 정리한 텍스트를 전달한다.

또한 한 번에 너무 많은 요청을 보내지 않도록 배치 크기와 동시 요청 수를 제한한다.  
이는 출력 누락, 파싱 오류, API 호출 오류를 줄이기 위한 설정이다.

| 설정값 | 설명 |
|---|---|
| `MIN_REVIEW_LEN` | 분석에 사용할 최소 리뷰 길이이다. |
| `MAX_REVIEW_CHARS` | LLM에 전달할 리뷰 본문 최대 글자 수이다. |
| `BATCH_SIZE` | 한 번의 LLM 요청에 묶어 보낼 리뷰 수이다. |
| `MAX_CONCURRENT` | 동시에 실행할 LLM 요청 수이다. |
| `MAX_RETRIES` | LLM 호출 실패 시 재시도할 횟수이다. |
| `REQUEST_SLEEP_SEC` | 요청 사이 대기 시간이다. |
| `CHUNK_SIZE` | 한 번에 처리할 비동기 작업 묶음 크기이다. |

## 1-5. 저장 및 비용 추정 설정

LLM 분석 결과는 후속 전처리와 대시보드 활용을 위해 JSON과 CSV 형태로 저장한다.

비용 추정 설정은 실제 과금액을 확정하는 값이 아니라, 실행 후 토큰 사용량을 기준으로 대략적인 비용을 확인하기 위한 참고값이다.

In [4]:
# ============================================================
# 실행 여부
# ============================================================
RUN_LLM = False
RESET_CHECKPOINT = False
RUN_CHECK_CELLS = True

# ============================================================
# 분석 대상 설정
# ============================================================
# 이번 버전은 특정 장르 한정이 아니라,
# 설정한 필터를 통과한 전체 게임의 D0-D30 초기 리뷰를 대상으로 한다.

ANALYSIS_FILTERS = {
    "appids": [],
    "game_name_contains": [],

    # 장르 제한 없음
    "genres": [],

    "categories": [],
    "tags": [],
    "release_date_from": "2023-01-01",
    "release_date_to": None,

    # 리뷰 조건
    "languages": ["english"],
    "steam_labels": ["positive", "negative"],
    "review_date_from": None,
    "review_date_to": None,

    # 출시 초기 리뷰만 사용
    "release_periods": ["D0-D30"],

    # 신뢰도 조건
    "steam_purchase_only": True,
    "exclude_received_for_free": True,
    "exclude_early_access_reviews": True,

    # 너무 짧거나 의미가 약한 리뷰 제외
    "meaningful_review_only": True,
}

# ============================================================
# 샘플링 설정
# ============================================================
TEST_N = None
RANDOM_STATE = 42

# 게임별 최대 리뷰 수
REVIEWS_PER_GAME = 100

# 전체 LLM 분석 리뷰 상한
MAX_TOTAL_REVIEWS = 5000

# 출시 전 리스크 점검을 위해 부정 리뷰를 우선 확보한다.
# NEGATIVE_SAMPLE_RATIO=0.8은 게임별 샘플에서 부정 리뷰를 최대 80%까지 확보하려는 목표 비율이다.
# 단, 실제 비율은 게임별 부정 리뷰 후보 수에 따라 달라질 수 있다.
SAMPLE_MODE = "negative_heavy_by_steam_label"
NEGATIVE_SAMPLE_RATIO = 0.8

# ============================================================
# LLM 입력 텍스트 설정
# ============================================================
MIN_REVIEW_LEN = 20
MAX_REVIEW_CHARS = 1200

# ============================================================
# LLM 호출 설정
# ============================================================
BATCH_SIZE = 3
MAX_CONCURRENT = 1
MAX_RETRIES = 3
REQUEST_SLEEP_SEC = 1
CHUNK_SIZE = MAX_CONCURRENT * 3

# ============================================================
# 저장 옵션
# ============================================================
SAVE_RESULT_JSON = True


# ============================================================
# 비용 추정 옵션
# 실제 Vertex AI 과금과 다를 수 있으므로 참고용으로만 사용한다.
# ============================================================
# 아래 단가는 대략적인 참고용이다.
# 실제 비용 확인은 Google Cloud Billing / Vertex AI pricing 기준으로 확인한다.
INPUT_PRICE_PER_1M = 0.30
OUTPUT_PRICE_PER_1M = 2.50
USD_TO_KRW = 1500


print("RUN_LLM:", RUN_LLM)
print("분석 대상: 리뷰 데이터가 있는 게임 전체")
print("SAMPLE_MODE:", SAMPLE_MODE)
print("REVIEWS_PER_GAME:", REVIEWS_PER_GAME)
print("MAX_TOTAL_REVIEWS:", MAX_TOTAL_REVIEWS)


RUN_LLM: False
분석 대상: 리뷰 데이터가 있는 게임 전체
SAMPLE_MODE: negative_heavy_by_steam_label
REVIEWS_PER_GAME: 100
MAX_TOTAL_REVIEWS: 5000


# 2. 공통 함수

In [5]:
# pandas/numpy/Pydantic 객체를 JSON/CSV 저장 가능한 기본 타입으로 변환한다.
def to_serializable(obj):
    """JSON 저장이 어려운 pandas/numpy/Pydantic 타입을 기본 Python 타입으로 변환한다."""
    if isinstance(obj, BaseModel):
        return to_serializable(obj.model_dump())
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, datetime):
        return obj.isoformat()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def safe_json_dumps(obj):
    """리스트/딕셔너리 컬럼을 CSV에 안전하게 저장하기 위한 JSON 문자열 변환 함수."""
    safe_obj = to_serializable(obj)
    return json.dumps(safe_obj, ensure_ascii=False)


def parse_issue_tags(value):
    """
    llm_issue_tags 값을 안정적으로 리스트[dict] 형태로 변환한다.

    가능한 입력 형태:
    - 이미 list인 경우
    - JSON 문자열: [{"category": "..."}]
    - Python repr 문자열: [{'category': '...'}]
    - 비어 있는 값 / NaN
    """
    if value is None:
        return []

    if isinstance(value, float) and pd.isna(value):
        return []

    if isinstance(value, list):
        tags = value
    elif isinstance(value, str):
        text = value.strip()
        if not text or text.lower() in ["nan", "none", "null"]:
            return []

        try:
            tags = json.loads(text)
        except Exception:
            try:
                tags = ast.literal_eval(text)
            except Exception:
                return []
    else:
        return []

    if not isinstance(tags, list):
        return []

    normalized = []
    for tag in tags:
        if isinstance(tag, BaseModel):
            tag = tag.model_dump()
        if isinstance(tag, dict):
            normalized.append({
                "category": tag.get("category"),
                "sentiment": tag.get("sentiment"),
                "evidence": tag.get("evidence"),
            })

    return normalized


def ensure_columns(df, columns):
    """DataFrame에 필요한 컬럼이 없으면 빈 컬럼을 추가하고, 지정 순서대로 정렬한다."""
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = pd.NA
    return out[columns].copy()


def save_csv_safely(df, path, columns=None, json_cols=None, encoding="utf-8-sig"):
    """
    CSV 저장 공통 함수.
    - 결과가 비어 있어도 헤더가 있는 CSV를 저장한다.
    - 리스트/딕셔너리 컬럼은 JSON 문자열로 변환한다.
    - 저장 폴더가 없으면 생성한다.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if df is None:
        df = pd.DataFrame()

    out = df.copy()

    if columns is not None:
        out = ensure_columns(out, columns)

    for col in (json_cols or []):
        if col in out.columns:
            out[col] = out[col].apply(lambda x: safe_json_dumps(parse_issue_tags(x)))

    out.to_csv(path, index=False, encoding=encoding)
    return out

# checkpoint 저장/불러오기 함수
def load_checkpoint(path=CHECKPOINT_PATH):
    """이전 LLM 분석 checkpoint를 불러온다."""
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_checkpoint(results, path=CHECKPOINT_PATH):
    """현재까지의 LLM 분석 결과를 checkpoint JSON으로 저장한다."""
    safe_results = to_serializable(results)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_results, f, ensure_ascii=False, indent=2)


def load_existing_results():
    """RUN_LLM=False일 때 기존 결과를 읽는다."""
    if RESULT_JSON_PATH.exists():
        with open(RESULT_JSON_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    return load_checkpoint(CHECKPOINT_PATH)


# 비용/토큰 사용량 확인 함수
def print_cost_report(input_tokens, output_tokens, requests, checkpoint_count, to_process_count):
    """토큰 사용량과 예상 비용을 출력한다."""
    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    total_cost = input_cost + output_cost

    print("=" * 60)
    print("토큰 사용량 / 예상 비용")
    print("=" * 60)
    print(f"요청 수: {requests:,}")
    print(f"checkpoint에서 불러온 리뷰 수: {checkpoint_count:,}")
    print(f"이번 실행에서 새로 처리할 리뷰 수: {to_process_count:,}")
    print(f"입력 토큰: {input_tokens:,}")
    print(f"출력 토큰: {output_tokens:,}")
    print(f"예상 비용(USD): ${total_cost:,.6f}")
    print(f"예상 비용(KRW): ₩{total_cost * USD_TO_KRW:,.0f}")
    print("=" * 60)

# PydanticAI 사용량 추출 함수
def extract_usage_tokens(result):
    """
    PydanticAI 결과 객체에서 토큰 사용량을 안전하게 추출한다.

    PydanticAI/모델 버전에 따라 usage 속성명이 조금 다를 수 있어
    여러 후보 이름을 순서대로 확인한다.
    """
    input_tokens = 0
    output_tokens = 0

    try:
        usage = result.usage()

        input_tokens = (
            getattr(usage, "input_tokens", None)
            or getattr(usage, "request_tokens", None)
            or getattr(usage, "prompt_tokens", None)
            or 0
        )

        output_tokens = (
            getattr(usage, "output_tokens", None)
            or getattr(usage, "response_tokens", None)
            or getattr(usage, "completion_tokens", None)
            or 0
        )

    except Exception:
        pass

    return int(input_tokens or 0), int(output_tokens or 0)

# 필터링/문자열 검색 헬퍼 함수
def print_filter_step(log_rows, step, before, after):
    """필터링 단계별 행 수 변화를 기록한다."""
    removed = before - after
    removed_rate = removed / before if before else 0
    log_rows.append({
        "step": step,
        "before_rows": before,
        "after_rows": after,
        "removed_rows": removed,
        "removed_rate": removed_rate,
    })
    print(f"{step}: {before:,} -> {after:,} / 제거 {removed:,} ({removed_rate:.2%})")


def contains_any_text(text, keywords):
    """문자열에 키워드 중 하나라도 포함되어 있는지 확인한다."""
    if not keywords:
        return True
    if pd.isna(text):
        return False
    text = str(text).lower()
    return any(str(keyword).lower() in text for keyword in keywords)


def pydantic_list_to_dicts(items):
    """Pydantic 객체 리스트를 CSV/JSON 저장 가능한 dict 리스트로 변환한다."""
    if items is None:
        return []
    if not isinstance(items, list):
        return []

    converted = []
    for item in items:
        if isinstance(item, BaseModel):
            converted.append(item.model_dump())
        elif isinstance(item, dict):
            converted.append(item)
    return converted


# 3. 전처리 후보 데이터 로드

In [6]:
# 전처리 후보 데이터 로드
df_candidates = pd.read_csv(PREPROCESSED_REVIEWS_PATH)

for col in ["review_datetime", "release_date"]:
    if col in df_candidates.columns:
        df_candidates[col] = pd.to_datetime(df_candidates[col], errors="coerce")

if "recommendationid" in df_candidates.columns:
    df_candidates["recommendationid"] = df_candidates["recommendationid"].astype(str)

print("전처리 후보 리뷰 수:", len(df_candidates))
print("전처리 후보 게임 수:", df_candidates["appid"].nunique())
display(df_candidates.head())


C:\Users\joon5\AppData\Local\Temp\ipykernel_45080\1519641675.py:2: DtypeWarning: Columns (0: top_steam_tags_text, 1: hist_first_date, 2: hist_last_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df_candidates = pd.read_csv(PREPROCESSED_REVIEWS_PATH)


전처리 후보 리뷰 수: 160644
전처리 후보 게임 수: 192


,recommendationid,appid,game_name,language,review_datetime,release_date,days_from_release,release_period,release_period_detail,steam_label_text,voted_up,playtime_at_review_hours,playtime_forever_hours,playtime_stage,votes_up,weighted_vote_score,steam_purchase,received_for_free,written_during_early_access,genres_text,categories_text,top_steam_tags_text,price,price_group,review_text_clean,review_len,is_meaningful_review,meaningless_reason,hist_total_reviews,hist_positive_reviews,hist_negative_reviews,hist_first_date,hist_last_date
0,18698790,324470,SinaRun,french,2015-10-26 18:10:33,2025-11-03,-3661,pre_release,pre_release,positive,True,1.250000,3.483333,early,2,0.523810,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,good game for this price,24,False,too_short,NaN,NaN,NaN,NaN,NaN
1,18699465,324470,SinaRun,english,2015-10-26 18:53:36,2025-11-03,-3661,pre_release,pre_release,positive,True,0.216667,0.216667,very_early,1,0.421372,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effect is too excessive \nSo level of difficulty is too hard for beginner ...",119,True,meaningful,NaN,NaN,NaN,NaN,NaN
2,18699648,324470,SinaRun,english,2015-10-26 19:03:37,2025-11-03,-3661,pre_release,pre_release,positive,True,12.666667,13.616667,late,5,0.500076,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,"this game is like a zen-garden, I love it! \n\npros:\n-it's very relaxing\n-good controls\n-awesome leveldesign\n-re...",387,True,meaningful,NaN,NaN,NaN,NaN,NaN
3,18700348,324470,SinaRun,english,2015-10-26 19:52:39,2025-11-03,-3661,pre_release,pre_release,positive,True,0.916667,7.483333,early,16,0.637511,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,Ever played Bhop? Surf? If so this games mechanics will feel Instantly similar too you. This game gives you such a r...,1100,True,meaningful,NaN,NaN,NaN,NaN,NaN
4,18701774,324470,SinaRun,english,2015-10-26 21:32:24,2025-11-03,-3661,pre_release,pre_release,positive,True,6.416667,8.666667,mid,4,0.495810,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,It's Lit,8,False,too_short,NaN,NaN,NaN,NaN,NaN


In [7]:
# ============================================================
# boolean 컬럼 타입 안정화
# ============================================================
# CSV를 다시 읽으면 True/False 값이 문자열로 들어오는 경우가 있다.
# 이후 필터링에서 == True, != True 조건을 안정적으로 사용하기 위해
# 주요 boolean 컬럼을 True / False / NaN 형태로 정리한다.

def normalize_bool_series(s):
    """문자열/숫자/bool 형태의 값을 True/False/NaN으로 정리한다."""
    text = s.astype(str).str.lower().str.strip()

    return pd.Series(
        np.select(
            [
                text.isin(["true", "1", "yes", "y"]),
                text.isin(["false", "0", "no", "n"]),
            ],
            [True, False],
            default=np.nan,
        ),
        index=s.index,
    )

bool_cols = [
    "voted_up",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "is_meaningful_review",
]

for bool_col in bool_cols:
    if bool_col in df_candidates.columns:
        df_candidates[bool_col] = normalize_bool_series(df_candidates[bool_col])

print("boolean 컬럼 정리 완료")
print(df_candidates[[col for col in bool_cols if col in df_candidates.columns]].dtypes)

boolean 컬럼 정리 완료
voted_up                       float64
steam_purchase                 float64
received_for_free              float64
written_during_early_access    float64
is_meaningful_review           float64
dtype: object


# 4. 분석 조건 필터링

In [8]:
# ANALYSIS_FILTERS에 입력한 조건을 실제 후보 데이터에 적용한다.
def apply_analysis_filters(df, filters):
    """LLM 실행 파일에서 분석 목적에 맞는 필터를 적용한다."""
    filtered = df.copy()
    log_rows = []

    # 0. 최소 리뷰 길이 필터
    if "review_len" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["review_len"] >= MIN_REVIEW_LEN]
        print_filter_step(log_rows, "최소 리뷰 길이 필터", before, len(filtered))

    # 0-1. 의미 있는 리뷰 필터
    if filters.get("meaningful_review_only") is True and "is_meaningful_review" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["is_meaningful_review"] == True]
        print_filter_step(log_rows, "의미 있는 리뷰 필터", before, len(filtered))

    # 1. 게임 직접 지정 필터
    if filters.get("appids"):
        before = len(filtered)
        appids = [int(x) for x in filters["appids"]]
        filtered = filtered[filtered["appid"].isin(appids)]
        print_filter_step(log_rows, "appid 필터", before, len(filtered))

    if filters.get("game_name_contains"):
        before = len(filtered)
        keywords = [str(x).lower() for x in filters["game_name_contains"]]
        filtered = filtered[filtered["game_name"].fillna("").str.lower().apply(lambda x: any(k in x for k in keywords))]
        print_filter_step(log_rows, "게임명 키워드 필터", before, len(filtered))

    # 2. 게임 속성 필터
    # 장르/카테고리/태그 텍스트에 지정한 키워드가 포함되는지 확인한다.
    if filters.get("genres"):
        before = len(filtered)
        filtered = filtered[filtered["genres_text"].apply(lambda x: contains_any_text(x, filters["genres"]))]
        print_filter_step(log_rows, "장르 필터", before, len(filtered))

    if filters.get("categories"):
        before = len(filtered)
        filtered = filtered[filtered["categories_text"].apply(lambda x: contains_any_text(x, filters["categories"]))]
        print_filter_step(log_rows, "카테고리 필터", before, len(filtered))

    if filters.get("tags"):
        before = len(filtered)
        filtered = filtered[filtered["top_steam_tags_text"].apply(lambda x: contains_any_text(x, filters["tags"]))]
        print_filter_step(log_rows, "태그 필터", before, len(filtered))

    if filters.get("release_date_from"):
        before = len(filtered)
        start = pd.to_datetime(filters["release_date_from"])
        filtered = filtered[filtered["release_date"] >= start]
        print_filter_step(log_rows, "출시일 시작 필터", before, len(filtered))

    if filters.get("release_date_to"):
        before = len(filtered)
        end = pd.to_datetime(filters["release_date_to"])
        filtered = filtered[filtered["release_date"] <= end]
        print_filter_step(log_rows, "출시일 종료 필터", before, len(filtered))

    # 3. 리뷰 조건 필터
    # 언어, Steam 라벨, 리뷰 작성일, 출시 기준 구간을 적용한다.
    if filters.get("languages") and "language" in filtered.columns:
        before = len(filtered)
        allowed = [x.lower() for x in filters["languages"]]
        filtered = filtered[filtered["language"].fillna("").str.lower().isin(allowed)]
        print_filter_step(log_rows, "언어 필터", before, len(filtered))

    if filters.get("steam_labels"):
        before = len(filtered)
        filtered = filtered[filtered["steam_label_text"].isin(filters["steam_labels"])]
        print_filter_step(log_rows, "Steam 라벨 필터", before, len(filtered))

    if filters.get("review_date_from"):
        before = len(filtered)
        start = pd.to_datetime(filters["review_date_from"])
        filtered = filtered[filtered["review_datetime"] >= start]
        print_filter_step(log_rows, "리뷰 작성일 시작 필터", before, len(filtered))

    if filters.get("review_date_to"):
        before = len(filtered)
        end = pd.to_datetime(filters["review_date_to"])
        filtered = filtered[filtered["review_datetime"] <= end]
        print_filter_step(log_rows, "리뷰 작성일 종료 필터", before, len(filtered))

    if filters.get("release_periods"):
        before = len(filtered)
        filtered = filtered[filtered["release_period"].isin(filters["release_periods"])]
        print_filter_step(log_rows, "출시 기준 리뷰 구간 필터", before, len(filtered))

    # 4. 리뷰 신뢰도 필터
    # 실제 구매 리뷰 위주로 보고, 무료 수령/얼리액세스 리뷰를 제외할 수 있다.
    if filters.get("steam_purchase_only") is True and "steam_purchase" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["steam_purchase"] == True]
        print_filter_step(log_rows, "Steam 구매 리뷰 필터", before, len(filtered))

    if filters.get("exclude_received_for_free") is True and "received_for_free" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["received_for_free"] != True]
        print_filter_step(log_rows, "무료 수령 리뷰 제외", before, len(filtered))

    if filters.get("exclude_early_access_reviews") is True and "written_during_early_access" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["written_during_early_access"] != True]
        print_filter_step(log_rows, "얼리액세스 리뷰 제외", before, len(filtered))

    return filtered.reset_index(drop=True), pd.DataFrame(log_rows)

# 분석 조건 적용
df_filtered, filter_log = apply_analysis_filters(df_candidates, ANALYSIS_FILTERS)

print("최종 필터링 리뷰 수:", len(df_filtered))
print("최종 필터링 게임 수:", df_filtered["appid"].nunique())

if RUN_CHECK_CELLS:
    display(filter_log)
    print("필터링 후 release_period 분포")
    display(df_filtered["release_period"].value_counts(dropna=False))
    print("필터링 후 Steam 라벨 분포")
    display(df_filtered["steam_label_text"].value_counts(dropna=False))


최소 리뷰 길이 필터: 160,644 -> 112,192 / 제거 48,452 (30.16%)
의미 있는 리뷰 필터: 112,192 -> 53,999 / 제거 58,193 (51.87%)
출시일 시작 필터: 53,999 -> 53,999 / 제거 0 (0.00%)
언어 필터: 53,999 -> 37,794 / 제거 16,205 (30.01%)
Steam 라벨 필터: 37,794 -> 37,794 / 제거 0 (0.00%)
출시 기준 리뷰 구간 필터: 37,794 -> 8,592 / 제거 29,202 (77.27%)
Steam 구매 리뷰 필터: 8,592 -> 8,592 / 제거 0 (0.00%)
무료 수령 리뷰 제외: 8,592 -> 8,569 / 제거 23 (0.27%)
얼리액세스 리뷰 제외: 8,569 -> 8,566 / 제거 3 (0.04%)
최종 필터링 리뷰 수: 8566
최종 필터링 게임 수: 163


,step,before_rows,after_rows,removed_rows,removed_rate
0,최소 리뷰 길이 필터,160644,112192,48452,0.301611
1,의미 있는 리뷰 필터,112192,53999,58193,0.518691
2,출시일 시작 필터,53999,53999,0,0.000000
3,언어 필터,53999,37794,16205,0.300098
4,Steam 라벨 필터,37794,37794,0,0.000000
5,출시 기준 리뷰 구간 필터,37794,8592,29202,0.772662
6,Steam 구매 리뷰 필터,8592,8592,0,0.000000
7,무료 수령 리뷰 제외,8592,8569,23,0.002677
8,얼리액세스 리뷰 제외,8569,8566,3,0.000350


필터링 후 release_period 분포


release_period
D0-D30    8566
Name: count, dtype: int64

필터링 후 Steam 라벨 분포


steam_label_text
positive    7453
negative    1113
Name: count, dtype: int64

# 5. 게임별 샘플링 및 LLM 입력 파일 저장

In [9]:
# 게임 1개에 대해 리뷰를 샘플링한다.
# - recent: 최신 리뷰 우선
# - random: 무작위
# - balanced_by_steam_label: 긍정/부정 리뷰를 가능하면 균형 있게 섞음
# - negative_heavy_by_steam_label: 부정 리뷰를 더 많이 포함하도록 섞음
def sample_one_game(group, reviews_per_game, mode, random_state):
    """
    게임 1개에 대해 리뷰를 샘플링하는 함수.
    샘플링 방식은 분석 목적에 따라 LLM 실행 파일에서 선택한다.
    """
    if reviews_per_game is None or len(group) <= reviews_per_game:
        return group

    if mode == "recent":
        return group.sort_values("review_datetime", ascending=False).head(reviews_per_game)

    if mode == "random":
        return group.sample(n=reviews_per_game, random_state=random_state)

    if mode == "balanced_by_steam_label":
        half = reviews_per_game // 2

        positive = group[group["steam_label_text"] == "positive"]
        negative = group[group["steam_label_text"] == "negative"]

        pos_n = min(len(positive), half)
        neg_n = min(len(negative), reviews_per_game - pos_n)

        pos_sample = positive.sample(n=pos_n, random_state=random_state) if pos_n > 0 else positive.head(0)
        neg_sample = negative.sample(n=neg_n, random_state=random_state) if neg_n > 0 else negative.head(0)

        sampled = pd.concat([pos_sample, neg_sample], axis=0)

        # 한쪽 라벨이 부족해서 목표 개수보다 적게 뽑힌 경우,
        # 부족한 수는 남은 리뷰에서 채운다.
        remain_n = reviews_per_game - len(sampled)
        if remain_n > 0:
            remain_pool = group.drop(index=sampled.index, errors="ignore")
            if len(remain_pool) > 0:
                add_n = min(remain_n, len(remain_pool))
                sampled = pd.concat([
                    sampled,
                    remain_pool.sample(n=add_n, random_state=random_state)
                ], axis=0)

        return sampled.sample(frac=1, random_state=random_state)

    if mode == "negative_heavy_by_steam_label":
        target_neg_n = int(np.ceil(reviews_per_game * NEGATIVE_SAMPLE_RATIO))

        positive = group[group["steam_label_text"] == "positive"]
        negative = group[group["steam_label_text"] == "negative"]

        neg_n = min(len(negative), target_neg_n)
        pos_n = min(len(positive), reviews_per_game - neg_n)

        neg_sample = negative.sample(n=neg_n, random_state=random_state) if neg_n > 0 else negative.head(0)
        pos_sample = positive.sample(n=pos_n, random_state=random_state) if pos_n > 0 else positive.head(0)

        sampled = pd.concat([neg_sample, pos_sample], axis=0)

        # 부정/긍정 리뷰가 부족해서 목표 개수보다 적게 뽑힌 경우,
        # 부족한 수는 남은 리뷰에서 채운다.
        remain_n = reviews_per_game - len(sampled)
        if remain_n > 0:
            remain_pool = group.drop(index=sampled.index, errors="ignore")
            if len(remain_pool) > 0:
                add_n = min(remain_n, len(remain_pool))
                sampled = pd.concat([
                    sampled,
                    remain_pool.sample(n=add_n, random_state=random_state)
                ], axis=0)

        return sampled.sample(frac=1, random_state=random_state)

    raise ValueError(f"지원하지 않는 SAMPLE_MODE입니다: {mode}")

# 게임별 샘플링 실행
def sample_reviews_per_game(df, reviews_per_game, mode, random_state):
    sampled_groups = []

    for _, group in df.groupby("appid", group_keys=False):
        sampled_groups.append(sample_one_game(group, reviews_per_game, mode, random_state))

    if not sampled_groups:
        return df.head(0)

    sampled = pd.concat(sampled_groups, axis=0).reset_index(drop=True)

    if MAX_TOTAL_REVIEWS is not None and len(sampled) > MAX_TOTAL_REVIEWS:
        sampled = sampled.sample(n=MAX_TOTAL_REVIEWS, random_state=random_state).reset_index(drop=True)

    if TEST_N is not None:
        sampled = sampled.head(TEST_N).copy()

    return sampled


df_sampled = sample_reviews_per_game(
    df_filtered,
    reviews_per_game=REVIEWS_PER_GAME,
    mode=SAMPLE_MODE,
    random_state=RANDOM_STATE,
)

# LLM에 전달할 텍스트 길이 제한은 LLM 실행 파일에서 적용한다.
df_sampled["review_text_for_llm"] = df_sampled["review_text_clean"].fillna("").astype(str).str.slice(0, MAX_REVIEW_CHARS)

# LLM 입력 파일 컬럼 구성
# 프롬프트에 필요한 리뷰/게임 정보와 후속 결과 매칭에 필요한 키만 남긴다.
LLM_INPUT_COLUMNS = [
    # 원본 리뷰 식별 정보
    "recommendationid",              # 리뷰 고유 ID입니다. Steam 리뷰 1개를 구분하는 식별자입니다.
    "appid",                         # Steam 게임 고유 ID입니다. 어떤 게임의 리뷰인지 구분할 때 사용합니다.
    "game_name",                     # 게임 이름입니다. appid만 보면 알아보기 어려우므로 함께 전달합니다.
    "language",                      # 리뷰 작성 언어입니다. 현재 분석에서는 영어 리뷰 필터링 여부 확인에 사용합니다.

    # 리뷰 작성 시점 / 출시 후 구간 정보
    "review_datetime",               # 리뷰 작성 일시입니다. 출시 후 어느 시점의 반응인지 확인할 때 사용합니다.
    "release_date",                  # 게임 출시일입니다. 리뷰 작성일과 비교해 출시 후 경과일을 계산할 때 사용합니다.
    "days_from_release",             # 출시일 기준 리뷰 작성일까지 지난 일수입니다.
    "release_period",                # 출시 후 기간 구간입니다. 예: D0-D7, D8-D30 등입니다.
    "release_period_detail",         # 출시 후 구간을 더 세부적으로 나눈 값입니다. 초기 반응을 더 자세히 볼 때 사용합니다.

    # Steam 원본 라벨 / 리뷰 메타 정보
    "steam_label_text",              # Steam 추천 여부를 positive/negative 같은 문자열로 바꾼 값입니다.
    "voted_up",                      # Steam 원본 추천 여부입니다. True면 추천, False면 비추천 리뷰입니다.
    "playtime_at_review_hours",      # 리뷰 작성 시점의 플레이타임입니다. 짧은 플레이 후 부정 리뷰인지 확인할 수 있습니다.
    "playtime_stage",                # 플레이타임을 구간화한 값입니다. 초반/중반/장기 플레이 리뷰를 구분할 때 사용합니다.
    "votes_up",                      # 해당 리뷰가 받은 '유용함' 투표 수입니다. 리뷰 영향력이나 신뢰도 참고용입니다.
    "weighted_vote_score",           # Steam에서 제공하는 리뷰 가중 점수입니다. 리뷰 노출/신뢰도 참고용입니다.
    "received_for_free",             # 무료로 받은 게임인지 여부입니다. 일반 구매자 반응과 구분할 때 사용합니다.
    "written_during_early_access",   # 얼리액세스 기간에 작성된 리뷰인지 여부입니다.

    # 게임 메타 정보
    "price",                         # 게임 가격입니다. 가격대별 반응 분석에 사용합니다.
    "price_group",                   # 가격을 구간으로 나눈 값입니다. 가격대별 비교에 사용합니다.
    "genres_text",                   # 게임 장르 목록입니다. 장르별 반응 분석에 사용합니다.
    "categories_text",               # 게임 카테고리 목록입니다. 싱글/멀티/협동 여부 분석에 사용합니다.
    "top_steam_tags_text",           # 주요 Steam 태그 목록입니다. 태그별 반응 비교에 사용합니다.

    # LLM 입력 텍스트 / 유효성 판단 정보
    "review_text_for_llm",           # LLM에 실제로 전달할 리뷰 본문입니다. 전처리된 텍스트를 사용합니다.
    "is_meaningful_review",          # 분석에 의미 있는 리뷰인지 여부입니다. 무의미한 리뷰를 제외하거나 별도 확인할 때 사용합니다.
    "meaningless_reason",            # 무의미한 리뷰로 판단된 이유입니다. 예: 짧은 문장, 이모지만 존재, 내용 없음 등입니다.
]

input_cols = [c for c in LLM_INPUT_COLUMNS if c in df_sampled.columns]
df_for_llm = df_sampled[input_cols].copy()

# 최종 LLM 입력 파일과 필터 로그를 저장한다.
# 이 파일을 보면 어떤 리뷰가 실제 분석 대상이 되었는지 재현할 수 있다.
df_for_llm.to_csv(LLM_INPUT_PATH, index=False, encoding="utf-8-sig")
filter_log.to_csv(FILTER_LOG_PATH, index=False, encoding="utf-8-sig")

# 게임별로 최종 샘플 리뷰 수와 라벨 분포를 요약한다.
sample_summary = (
    df_for_llm
    .groupby(["appid", "game_name"], as_index=False)
    .agg(
        sampled_review_count=("recommendationid", "count"),
        positive_count=("steam_label_text", lambda x: (x == "positive").sum()),
        negative_count=("steam_label_text", lambda x: (x == "negative").sum()),
        first_review_datetime=("review_datetime", "min"),
        last_review_datetime=("review_datetime", "max"),
    )
    .sort_values("sampled_review_count", ascending=False)
)
sample_summary.to_csv(SAMPLE_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("LLM 입력 리뷰 수:", len(df_for_llm))
print("LLM 입력 게임 수:", df_for_llm["appid"].nunique())
print("LLM 입력 저장:", LLM_INPUT_PATH)
print("필터 로그 저장:", FILTER_LOG_PATH)
print("샘플 요약 저장:", SAMPLE_SUMMARY_PATH)

display(df_for_llm.head())
display(sample_summary.head())


LLM 입력 리뷰 수: 4429
LLM 입력 게임 수: 163
LLM 입력 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_input_reviews.csv
필터 로그 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_input_filter_log.csv
샘플 요약 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_input_sample_summary.csv


,recommendationid,appid,game_name,language,review_datetime,release_date,days_from_release,release_period,release_period_detail,steam_label_text,voted_up,playtime_at_review_hours,playtime_stage,votes_up,weighted_vote_score,received_for_free,written_during_early_access,price,price_group,genres_text,categories_text,top_steam_tags_text,review_text_for_llm,is_meaningful_review,meaningless_reason
0,144515690,571740,Golf It!,english,2023-08-18 21:03:00,2023-08-18,0,D0-D30,D0-D7,positive,1.0,2.916667,mid,6,0.563222,0.0,0.0,4.49,0-5,"Casual, Indie, Simulation, Sports","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...","Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor",I love the game and now that its out of early acces it is even better.,1.0,meaningful
1,144516841,571740,Golf It!,english,2023-08-18 21:23:34,2023-08-18,0,D0-D30,D0-D7,negative,0.0,1.533333,early,2,0.492600,0.0,0.0,4.49,0-5,"Casual, Indie, Simulation, Sports","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...","Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor",The only golf game to gets golf WRONG.. The way you hit the ball is too inconsistent in comparison to nearly every o...,1.0,meaningful
2,144522911,571740,Golf It!,english,2023-08-18 23:21:37,2023-08-18,0,D0-D30,D0-D7,positive,1.0,1.000000,early,0,0.500000,0.0,0.0,4.49,0-5,"Casual, Indie, Simulation, Sports","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...","Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","The game is INSANELY fun, but for whatever reason, I was hitting the ball and not moving. I'm not sure if it was jus...",1.0,meaningful
3,144527817,571740,Golf It!,english,2023-08-19 01:06:16,2023-08-18,1,D0-D30,D0-D7,positive,1.0,53.050000,late,1,0.500000,0.0,0.0,4.49,0-5,"Casual, Indie, Simulation, Sports","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...","Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor",the $9 Bucks i have spent. this game is packed with features. from Official Maps to workshop maps. there is thousand...,1.0,meaningful
4,144536439,571740,Golf It!,english,2023-08-19 04:08:08,2023-08-18,1,D0-D30,D0-D7,positive,1.0,131.216667,late,1,0.500000,0.0,0.0,4.49,0-5,"Casual, Indie, Simulation, Sports","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...","Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","A wonderful game from a caring developer. So many hours were put into making this a replayable, enjoyable experience.",1.0,meaningful


,appid,game_name,sampled_review_count,positive_count,negative_count,first_review_datetime,last_review_datetime
6,824600,HROT,100,87,13,2023-05-16 19:15:54,2023-06-15 09:29:32
5,695330,SEASON: A letter to the future,100,92,8,2023-01-31 19:18:44,2023-03-02 07:12:14
3,619820,Heroes of Hammerwatch II,100,20,80,2025-01-15 05:23:17,2025-02-10 14:59:07
29,1603180,Mining Mechs,100,93,7,2023-10-27 14:13:58,2023-11-26 03:31:33
25,1536620,Galactic Glitch,100,94,6,2025-06-03 18:14:06,2025-06-21 20:19:31


# 6. PydanticAI 출력 스키마 정의

- 결과 컬럼이 매번 달라지는 것을 방지한다.
- JSON 파싱 오류를 줄인다.
- 후속 집계표를 안정적으로 만들 수 있다.
- `category` 목록은 이후 이슈 집계와 시각화의 기준이 된다.

## `llm_urgency_candidate` 해석 기준

이 단계의 `llm_urgency_candidate`는 LLM이 리뷰 문맥을 보고 판단한 **시급도 후보**다.  
예를 들어 크래시, 저장 오류, 진행 불가처럼 출시 직후 경험에 큰 영향을 줄 수 있는 내용은 높게 분류될 수 있다.

다만 이 값은 **최종 개선 우선순위가 아니다.**  
최종 체크리스트 우선순위는 이후 코드에서 여러 게임에서의 반복성, 부정 맥락, 조건 내 발생 비율 등을 함께 고려해 데이터 기준으로 산정한다.


| 설정값 | 의미 |
|---|---|
| `bug` | 버그 |
| `optimization` | 최적화 |
| `performance` | 성능/프레임 |
| `crash` | 튕김/실행 불가 |
| `control` | 조작감 |
| `balance` | 밸런스 |
| `difficulty` | 난이도 |
| `content_volume` | 콘텐츠 양 |
| `story` | 스토리 |
| `translation_localization` | 번역/현지화 |
| `ui_ux` | UI/UX |
| `price_value` | 가격 대비 가치 |
| `multiplayer_network` | 멀티/서버 |
| `save_progression` | 저장/진행도 |
| `graphics_audio` | 그래픽/사운드 |
| `gameplay_loop` | 핵심 재미/반복 구조 |
| `monetization` | 과금/DLC |
| `developer_communication` | 개발자 소통 |
| `positive_praise` | 전반적 칭찬 |
| `progression_grind` | 성장/노가다 |
| `other` | 기타 |

In [10]:
# ============================================================
# LLM 출력 스키마
# ============================================================
# PydanticAI의 output_type으로 사용할 Pydantic 모델이다.
# 이 스키마를 기준으로 LLM 출력이 구조화되어 들어온다.
# ============================================================

class IssueTag(BaseModel):
    category: Literal[
        "bug",                          # 버그, 오류, 비정상 동작
        "optimization",                 # 최적화 전반, 렉, 로딩, 프레임 저하
        "performance",                  # 성능, 사양, 프레임 관련 문제
        "crash",                        # 튕김, 실행 불가, 강제 종료
        "control",                      # 조작감, 키 설정, 컨트롤러 문제
        "balance",                      # 밸런스, 캐릭터/무기/시스템 불균형
        "difficulty",                   # 난이도 관련 불만/칭찬
        "content_volume",               # 콘텐츠 양 부족/풍부함
        "story",                        # 스토리, 서사, 캐릭터, 세계관
        "translation_localization",     # 번역, 현지화, 언어 지원 문제
        "ui_ux",                        # UI, UX, 메뉴, 정보 전달 문제
        "price_value",                  # 가격 대비 가치, 할인, 볼륨 대비 가격
        "multiplayer_network",          # 멀티플레이, 서버, 매칭, 네트워크
        "save_progression",             # 저장, 진행도, 체크포인트, 세이브 손실
        "graphics_audio",               # 그래픽, 사운드, 연출, 아트 스타일
        "gameplay_loop",                # 핵심 재미, 반복 구조, 전투/플레이 흐름
        "monetization",                 # 과금, DLC, BM, 유료 요소
        "developer_communication",      # 개발자 소통, 패치 대응, 공지
        "positive_praise",              # 구체 이슈라기보다 전반적 칭찬
        "progression_grind",            # 성장, 반복 플레이, 노가다 구조
        "other",                        # 위 범주로 분류하기 어려운 기타 이슈
    ] = Field(description="리뷰에서 언급된 세부 이슈 카테고리")

    # sentiment는 해당 이슈에 대한 감정 방향이다.
    # 리뷰 전체 감정이 아니라, 이 세부 이슈 하나에 대한 감정이다.
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="이 이슈에 대한 감정"
    )

    # evidence는 왜 이 카테고리/감정으로 판단했는지에 대한 짧은 근거다.
    # 원문 리뷰를 바탕으로 작성된다.
    evidence: str = Field(
        description="원문 리뷰를 바탕으로 한 짧은 판단 근거",
        min_length=1,
        max_length=160,
    )


class SteamReviewAnalysis(BaseModel):
    # 입력 리뷰 ID를 그대로 반환한다.
    recommendationid: str = Field(description="입력 리뷰 ID 그대로 반환")

    # LLM이 리뷰 본문만 보고 판단한 감정이다.
    # Steam의 voted_up과 다를 수 있다.
    llm_sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="리뷰 본문 기준 감정"
    )

    # 감정을 1~5점으로 수치화한 값이다.
    # 1은 매우 부정, 3은 중립, 5는 매우 긍정으로 해석한다.
    sentiment_score: int = Field(
        ge=1,
        le=5,
        description="1=매우 부정, 3=중립, 5=매우 긍정",
    )

    llm_primary_issue: Literal[
        "bug",                          # 버그, 오류, 비정상 동작
        "optimization",                 # 최적화 전반, 렉, 로딩, 프레임 저하
        "performance",                  # 성능, 사양, 프레임 관련 문제
        "crash",                        # 튕김, 실행 불가, 강제 종료
        "control",                      # 조작감, 키 설정, 컨트롤러 문제
        "balance",                      # 밸런스, 캐릭터/무기/시스템 불균형
        "difficulty",                   # 난이도 관련 불만/칭찬
        "content_volume",               # 콘텐츠 양 부족/풍부함
        "story",                        # 스토리, 서사, 캐릭터, 세계관
        "translation_localization",     # 번역, 현지화, 언어 지원 문제
        "ui_ux",                        # UI, UX, 메뉴, 정보 전달 문제
        "price_value",                  # 가격 대비 가치, 할인, 볼륨 대비 가격
        "multiplayer_network",          # 멀티플레이, 서버, 매칭, 네트워크
        "save_progression",             # 저장, 진행도, 체크포인트, 세이브 손실
        "graphics_audio",               # 그래픽, 사운드, 연출, 아트 스타일
        "gameplay_loop",                # 핵심 재미, 반복 구조, 전투/플레이 흐름
        "monetization",                 # 과금, DLC, BM, 유료 요소
        "developer_communication",      # 개발자 소통, 패치 대응, 공지
        "positive_praise",              # 구체 이슈라기보다 전반적 칭찬
        "progression_grind",            # 성장, 반복 플레이, 노가다 구조
        "other",                        # 위 범주로 분류하기 어려운 기타 이슈
    ] = Field(description="리뷰의 대표 이슈")

    # 리뷰 안에서 발견된 여러 세부 이슈 목록이다.
    llm_issue_tags: List[IssueTag] = Field(
        default_factory=list,
        description="리뷰에서 발견된 세부 이슈 목록",
    )

    # 개발사 입장에서 대응 긴급도다.
    llm_urgency_candidate: Literal["low", "medium", "high"] = Field(
        description="LLM이 리뷰 문맥을 보고 분류한 시급도 후보. 최종 체크리스트 우선순위는 아님"
    )

    # 리뷰 핵심 내용 요약이다.
    llm_review_summary: str = Field(
        description="리뷰 핵심 내용 요약",
        min_length=5,
        max_length=220,
    )

    # 개발사 관점의 후속 액션이다.
    llm_suggested_action: str = Field(
        description="리뷰 내용을 바탕으로 정리한 개선 방향 후보",
        min_length=5,
        max_length=260,
    )


class BatchSteamReviewAnalysis(BaseModel):
    """한 번의 배치 요청에서 여러 리뷰 결과를 받을 수 있도록 감싸는 모델."""
    results: List[SteamReviewAnalysis] = Field(
        description="리뷰별 분석 결과 목록"
    )


ISSUE_KR_MAP = {
    "bug": "버그",
    "optimization": "최적화",
    "performance": "성능",
    "crash": "크래시",
    "control": "조작감",
    "balance": "밸런스",
    "difficulty": "난이도",
    "content_volume": "콘텐츠 분량",
    "story": "스토리",
    "translation_localization": "번역/현지화",
    "ui_ux": "UI/UX",
    "price_value": "가격/가치",
    "multiplayer_network": "멀티/네트워크",
    "save_progression": "저장/진행",
    "graphics_audio": "그래픽/사운드",
    "gameplay_loop": "게임플레이 루프",
    "monetization": "과금",
    "developer_communication": "개발사 소통",
    "positive_praise": "긍정 칭찬",
    "progression_grind": "성장/반복 노가다",
    "other": "기타",
}


# 7. 프롬프트 및 PydanticAI Agent 설정
LLM에게 어떤 역할을 부여할지, 어떤 모델 설정으로 호출할지 정한다.

| 설정값 | 의미 |
|---|---|
| `system_prompt` | LLM에게 부여하는 분석 기준과 제한 사항 |
| `temperature=0.0` | 같은 리뷰에 대해 가능한 한 일관적인 분류가 나오도록 설정 |
| `review_agent` | PydanticAI가 Vertex AI Gemini를 호출할 때 사용하는 Agent |



In [11]:
system_prompt = """
당신은 Steam 인디게임 리뷰 분석 전문가입니다.

각 리뷰에 대해 다음을 분류/작성하세요.
1. 리뷰 본문 기준 감정(llm_sentiment)
2. 가장 핵심적인 이슈(llm_primary_issue)
3. 세부 이슈(llm_issue_tags)
4. 리뷰 내용에 근거한 개발사 참고용 개선 방향 후보(llm_suggested_action)
5. 리뷰 문맥상 문제 강도 후보(llm_urgency_candidate)

주의:
- llm_urgency_candidate는 최종 개선 우선순위가 아니라, 리뷰 문맥에서 문제가 강하게 표현되었는지 확인하기 위한 보조 분류값입니다.
- 최종 출시 전 체크리스트 우선순위는 후속 코드에서 여러 게임 반복성, 조건 내 발생 비율, Steam 비추천 맥락 등을 기준으로 별도 계산합니다.

중요 규칙:
- recommendationid는 반드시 입력값 그대로 반환하세요.
- voted_up은 참고 정보일 뿐, 감정은 review 텍스트 기준으로 판단하세요.
- 추천 리뷰라도 불만이 많으면 mixed 또는 negative로 판단할 수 있습니다.
- 비추천 리뷰라도 장단점이 섞여 있으면 mixed로 판단할 수 있습니다.
- llm_issue_tags에는 실제로 언급된 것만 넣으세요.
- evidence는 리뷰 본문에 근거가 있는 짧은 표현 또는 요약으로 작성하세요.
- review가 매우 짧거나 밈/농담 위주면 과잉 해석하지 마세요.
- 분석할 정보가 부족한 리뷰는 llm_primary_issue를 other로 두고, llm_urgency_candidate는 low로 분류하세요.
- 리뷰 본문에 근거가 없는 개선 제안은 작성하지 말고 보수적으로 작성하세요.
- 게임 메타데이터와 태그는 맥락 참고용이며, 리뷰 본문에 없는 내용을 억지로 추론하지 마세요.
- llm_primary_issue가 positive_praise인 경우, llm_urgency_candidate는 원칙적으로 low로 분류하세요.
- 긍정 리뷰의 suggested_action은 문제 해결 지시가 아니라 유지/강화할 강점 중심으로 작성하세요.
- sentiment_score는 llm_sentiment와 일관되게 작성하세요.
- negative는 1~2, neutral은 3, mixed는 2~4 범위에서 맥락에 맞게, positive는 4~5로 작성하세요.
- llm_suggested_action은 최종 전략이 아니라, 리뷰 본문에 근거한 개발사 참고용 개선 방향 후보로 작성하세요.
- 인디게임 개발사가 실제로 참고할 수 있게 구체적으로 작성하되, 리뷰에 없는 위험을 새로 만들지 마세요.
- llm_urgency_candidate를 이용해 전체 우선순위를 정하거나, 리뷰에 없는 위험을 새로 추론하지 마세요.
"""

# Gemini 모델 세부 설정
# temperature=0.0으로 두어 같은 리뷰에 대해 가능한 한 일관적인 분류가 나오도록 한다.
review_settings = GoogleModelSettings(
    temperature=0.0,
)

# Vertex 모델이 정상 생성된 경우에만 PydanticAI Agent를 만든다.
# RUN_LLM=False로 기존 결과만 읽을 때는 Agent가 없어도 후처리 셀을 볼 수 있다.
if vertex_model is not None:
    review_agent = Agent(
        vertex_model,
        output_type=BatchSteamReviewAnalysis,
        system_prompt=system_prompt,
        retries=MAX_RETRIES,
        output_retries=3,
    )
else:
    review_agent = None

print("PydanticAI Agent 생성 여부:", "O" if review_agent is not None else "X")


PydanticAI Agent 생성 여부: O


# 8. 프롬프트 생성 함수


In [12]:
def build_batch_prompt(batch_df):
    """
    여러 개의 리뷰를 한 번에 LLM에게 보내기 위한 프롬프트 생성 함수.
    """
    blocks = []

    for _, row in batch_df.iterrows():
        block = f"""
[REVIEW]
recommendationid: {row["recommendationid"]}
appid: {row["appid"]}
game_name: {row.get("game_name", "")}
genres: {row.get("genres_text", "")}
categories: {row.get("categories_text", "")}
steam_top_tags: {row.get("top_steam_tags_text", "")}
release_date: {row.get("release_date", "")}
review_datetime: {row.get("review_datetime", "")}
days_from_release: {row.get("days_from_release", "")}
release_period: {row.get("release_period", "")}
language: {row.get("language", "")}
voted_up: {row.get("voted_up", "")}
steam_label_text: {row.get("steam_label_text", "")}
playtime_at_review_hours: {row.get("playtime_at_review_hours", "")}
playtime_stage: {row.get("playtime_stage", "")}
votes_up: {row.get("votes_up", "")}
weighted_vote_score: {row.get("weighted_vote_score", "")}
received_for_free: {row.get("received_for_free", "")}
written_during_early_access: {row.get("written_during_early_access", "")}

review:
{row.get("review_text_for_llm", "")}
[/REVIEW]
"""
        blocks.append(block)

    prompt = (
        f"다음 {len(batch_df)}개의 Steam 리뷰를 각각 분석해주세요.\\n"
        "반드시 입력된 recommendationid를 그대로 유지해서 반환하세요.\\n"
        "결과는 지정된 Pydantic 스키마에 맞게 반환하세요.\\n\\n"
        + "\\n".join(blocks)
    )

    return prompt


# 9. PydanticAI Vertex LLM 호출 함수

실제 Vertex AI Gemini 호출을 수행하는 함수

핵심 흐름은 다음과 같다.
1. 리뷰 배치를 프롬프트로 변환한다.
2. PydanticAI Agent로 Vertex AI Gemini를 호출한다.
3. LLM이 반환한 결과를 `recommendationid` 기준으로 원본 리뷰와 매칭한다.
4. 성공/누락/실패 결과를 모두 기록한다.
5. 중간 결과를 checkpoint에 저장해 실행 중단 시에도 복구할 수 있게 한다.

AI와 씨름한 결과물이라 이게 맞는건지는....

In [13]:
# 동시에 실행될 LLM 요청 수를 제한하기 위한 Semaphore
# MAX_CONCURRENT=1이면 한 번에 요청 1개만 실행
sem = asyncio.Semaphore(MAX_CONCURRENT)

# 리뷰 배치 1개를 PydanticAI + Vertex AI Gemini로 분석
# 1. batch_df를 LLM 프롬프트로 변환
# 2. Vertex AI Gemini 호출
# 3. Pydantic 구조로 받은 결과를 원본 리뷰와 매칭
# 4. 결과를 all_results에 추가
# 5. 실패하면 재시도하고, 최종 실패 시 실패 기록을 남김
async def analyze_batch(batch_df, all_results, stats, pbar):
    """
    리뷰 배치 1개를 PydanticAI + Vertex AI Gemini로 분석한다.
    """
    async with sem:
        prompt = build_batch_prompt(batch_df)

        for attempt in range(MAX_RETRIES):
            try:
                if review_agent is None:
                    raise RuntimeError(
                        "review_agent가 생성되지 않았습니다. "
                        ".env의 GOOGLE_CLOUD_PROJECT, gcloud ADC 인증, pydantic-ai 설치 여부를 확인하세요."
                    )

                result = await review_agent.run(
                    prompt,
                    model_settings=review_settings,
                )

                # PydanticAI가 스키마에 맞춰 구조화한 결과를 가져온다.
                output = result.output
                output_items = getattr(output, "results", [])

                # 토큰 사용량을 누적해 예상 비용을 확인
                input_tokens, output_tokens = extract_usage_tokens(result)
                stats["input_tokens"] += input_tokens
                stats["output_tokens"] += output_tokens
                stats["requests"] += 1

                # 입력 리뷰 ID와 LLM 반환 ID를 비교해 누락/오반환 여부를 확인
                input_ids = set(batch_df["recommendationid"].astype(str).tolist())
                matched_ids = set()

                for item in output_items:
                    rid = str(item.recommendationid)

                    # LLM이 입력에 없던 ID를 반환하면 무시한다.
                    if rid not in input_ids:
                        continue

                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    matched_ids.add(rid)

                    record = {
                        # 중요
                        # "llm_primary_issue": 부정/긍정 반응의 핵심 원인 분류입니다.
                        # "llm_urgency_candidate": LLM이 분류한 시급도 후보입니다. 최종 우선순위가 아니라 보조 참고용입니다.
                        # "llm_suggested_action": 리뷰 내용을 바탕으로 정리한 개선 방향 후보입니다.
                        "analysis_status": "success",  # LLM 분석이 정상적으로 완료된 리뷰임을 표시합니다.

                        # 원본 리뷰 식별 정보
                        "recommendationid": rid,  # 리뷰 고유 ID입니다. Steam 리뷰 1개를 구분하는 식별자입니다.
                        "appid": row.get("appid"),  # Steam 게임 고유 ID입니다. 어떤 게임의 리뷰인지 구분할 때 사용합니다.
                        "game_name": row.get("game_name", ""),  # 게임 이름입니다. appid만 보면 알아보기 어려우므로 함께 저장합니다.
                        "review_datetime": row.get("review_datetime", None),  # 리뷰 작성 일시입니다. 출시 후 반응 구간을 확인할 때 사용합니다.
                        "release_date": row.get("release_date", None),  # 게임 출시일입니다. 리뷰 작성 시점과 비교해 초기/장기 반응을 나눌 때 사용합니다.
                        "days_from_release": row.get("days_from_release", None),  # 출시일 기준 리뷰 작성일까지 지난 일수입니다.
                        "release_period": row.get("release_period", None),  # 출시 후 기간 구간입니다. 예: D0-D7, D8-D30 등입니다.

                        # Steam 라벨/리뷰 메타
                        "steam_label_text": row.get("steam_label_text", ""),  # Steam 추천 여부를 positive/negative 같은 문자열로 바꾼 값입니다.
                        "playtime_at_review_hours": row.get("playtime_at_review_hours", None),  # 리뷰 작성 시점의 플레이타임입니다. 짧은 플레이 후 부정 리뷰인지 확인할 수 있습니다.
                        "votes_up": row.get("votes_up", None),  # 해당 리뷰가 받은 '유용함' 투표 수입니다. 리뷰 영향력이나 신뢰도 참고용입니다.
                        "weighted_vote_score": row.get("weighted_vote_score", None),  # Steam에서 제공하는 리뷰 가중 점수입니다. 리뷰 노출/신뢰도 참고용입니다.

                        # LLM 분석 결과
                        "llm_sentiment": item.llm_sentiment,  # LLM이 판단한 리뷰 감정입니다. positive/negative/mixed/neutral 등으로 저장됩니다.
                        "sentiment_score": item.sentiment_score,  # LLM이 판단한 감정 점수입니다. 감정 강도를 수치로 비교할 때 사용합니다.
                        "llm_primary_issue": item.llm_primary_issue,  # LLM이 판단한 리뷰의 대표 이슈입니다. 버그/밸런스/콘텐츠/가격 등 주요 원인을 나타냅니다.
                        "llm_issue_tags": pydantic_list_to_dicts(item.llm_issue_tags),  # 리뷰 안에서 발견된 세부 이슈 태그 목록입니다. 한 리뷰에 여러 문제가 있을 수 있습니다.
                        "llm_urgency_candidate": item.llm_urgency_candidate,  # LLM이 분류한 시급도 후보입니다. 최종 우선순위가 아니라 보조 참고용입니다.
                        "llm_review_summary": item.llm_review_summary,  # LLM이 요약한 리뷰 핵심 내용입니다. 원문을 빠르게 파악하기 위한 요약입니다.
                        "llm_suggested_action": item.llm_suggested_action,  # LLM이 리뷰 내용을 바탕으로 정리한 개선 방향 후보입니다.
                    }

                    all_results.append(record)

                # LLM이 누락한 리뷰가 있으면 누락 기록을 남긴다.
                missing_ids = input_ids - matched_ids
                for rid in missing_ids:
                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    all_results.append({
                        "analysis_status": "missing_in_llm_output",
                        "recommendationid": rid,
                        "appid": row.get("appid"),
                        "game_name": row.get("game_name", ""),
                        "steam_label_text": row.get("steam_label_text", ""),
                        "llm_sentiment": None,
                        "sentiment_score": None,
                        "llm_primary_issue": None,
                        "llm_issue_tags": [],
                        "llm_urgency_candidate": None,
                        "llm_review_summary": None,
                        "llm_suggested_action": None,
                    })

                save_checkpoint(all_results)
                pbar.update(len(batch_df))
                return

            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    wait_sec = 2 ** attempt
                    print(f"배치 분석 실패, 재시도 {attempt + 1}/{MAX_RETRIES}: {e}")
                    await asyncio.sleep(wait_sec)
                else:
                    print(f"배치 최종 실패: {e}")

                    for _, row in batch_df.iterrows():
                        all_results.append({
                            "analysis_status": "failed",
                            "recommendationid": str(row.get("recommendationid")),
                            "appid": row.get("appid"),
                            "game_name": row.get("game_name", ""),
                            "steam_label_text": row.get("steam_label_text", ""),
                            "llm_sentiment": None,
                            "sentiment_score": None,
                            "llm_primary_issue": None,
                            "llm_issue_tags": [],
                            "llm_urgency_candidate": None,
                            "llm_review_summary": None,
                            "llm_suggested_action": None,
                            "error_message": str(e),
                        })

                    save_checkpoint(all_results)
                    pbar.update(len(batch_df))
                    return

# 전체 LLM 분석을 실행한다.
# 처리 흐름:
# 1. 기존 checkpoint를 읽는다.
# 2. 현재 분석 대상 ID만 checkpoint에서 유지한다.
# 3. 이미 처리된 recommendationid는 제외한다.
# 4. 남은 리뷰를 BATCH_SIZE 단위로 나눈다.
# 5. CHUNK_SIZE 단위로 비동기 요청을 실행한다.
# 6. 중간 결과는 checkpoint에 계속 저장한다.
async def run_analysis(df):
    """
    전체 LLM 분석을 실행한다.
    """
    if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
        CHECKPOINT_PATH.unlink()
        print("기존 checkpoint 삭제:", CHECKPOINT_PATH)

    all_results = load_checkpoint()
    all_results = list(all_results)

    # 현재 분석 대상 ID만 checkpoint에서 유지한다.
    target_ids = set(df["recommendationid"].astype(str))
    all_results = [
        row for row in all_results
        if str(row.get("recommendationid")) in target_ids
    ]

    done_ids = {
        str(row.get("recommendationid"))
        for row in all_results
        if row.get("analysis_status") in ["success", "missing_in_llm_output", "failed"]
    }

    to_process = df[~df["recommendationid"].astype(str).isin(done_ids)].copy()

    stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
        "checkpoint_count": len(done_ids),
        "to_process_count": len(to_process),
    }

    if len(to_process) == 0:
        print("새로 처리할 리뷰가 없습니다. checkpoint 또는 기존 결과를 사용합니다.")
        return all_results, stats

    batches = [
        to_process.iloc[i:i + BATCH_SIZE]
        for i in range(0, len(to_process), BATCH_SIZE)
    ]

    with tqdm(total=len(to_process), desc="PydanticAI Vertex 리뷰 분석 진행") as pbar:
        for start in range(0, len(batches), CHUNK_SIZE):
            chunk = batches[start:start + CHUNK_SIZE]

            tasks = [
                analyze_batch(batch_df, all_results, stats, pbar)
                for batch_df in chunk
            ]

            await asyncio.gather(*tasks)

            if REQUEST_SLEEP_SEC > 0:
                await asyncio.sleep(REQUEST_SLEEP_SEC)

    return all_results, stats


# 10. LLM 분석 실행

`RUN_LLM` 설정에 따라 실제 LLM 호출 여부가 달라진다.

- `RUN_LLM=True`: Vertex AI Gemini를 실제 호출한다.
- `RUN_LLM=False`: 기존 JSON/checkpoint 결과를 읽어 후처리만 수행한다.

실행 후에는 토큰 사용량과 예상 비용을 출력한다.

In [14]:
if RUN_LLM:
    results, stats = await run_analysis(df_for_llm)
else:
    results = load_existing_results()
    stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
        "checkpoint_count": len(results),
        "to_process_count": 0,
    }

print_cost_report(
    input_tokens=stats["input_tokens"],
    output_tokens=stats["output_tokens"],
    requests=stats["requests"],
    checkpoint_count=stats["checkpoint_count"],
    to_process_count=stats["to_process_count"],
)

safe_results = to_serializable(results)
df_result_raw = pd.DataFrame(safe_results)
if not RUN_LLM and len(df_result_raw) == 0:
    raise FileNotFoundError(
        "RUN_LLM=False인데 기존 LLM 결과 JSON/checkpoint를 찾지 못했습니다. "
        "빈 결과 파일로 덮어쓰는 것을 방지하기 위해 실행을 중단합니다. "
        "RESULT_JSON_PATH 또는 CHECKPOINT_PATH를 확인하세요."
    )

print("분석 결과 행 수:", len(df_result_raw))
display(df_result_raw.head())


토큰 사용량 / 예상 비용
요청 수: 0
checkpoint에서 불러온 리뷰 수: 4,428
이번 실행에서 새로 처리할 리뷰 수: 0
입력 토큰: 0
출력 토큰: 0
예상 비용(USD): $0.000000
예상 비용(KRW): ₩0
분석 결과 행 수: 4428


,analysis_status,recommendationid,appid,game_name,review_datetime,release_date,days_from_release,release_period,steam_label_text,playtime_at_review_hours,votes_up,weighted_vote_score,llm_sentiment,sentiment_score,llm_primary_issue,llm_issue_tags,llm_urgency_candidate,llm_review_summary,llm_suggested_action,steam_llm_sentiment_relation,error_message
0,success,144515690,571740,Golf It!,2023-08-18T21:03:00,2023-08-18T00:00:00,0.0,D0-D30,positive,2.916667,6.0,0.563222,positive,5.0,positive_praise,"[{'category': 'positive_praise', 'sentiment': 'positive', 'evidence': 'I love the game and now that its out of early...",low,정식 출시 이후 게임이 더욱 좋아졌으며 매우 만족함.,현재의 게임 플레이 경험과 정식 출시 버전의 완성도를 유지 및 강화할 것.,exact_match,None
1,success,144516841,571740,Golf It!,2023-08-18T21:23:34,2023-08-18T00:00:00,0.0,D0-D30,negative,1.533333,2.0,0.492600,negative,1.0,gameplay_loop,"[{'category': 'gameplay_loop', 'sentiment': 'negative', 'evidence': 'The way you hit the ball is too inconsistent in...",high,골프 게임의 핵심인 타격 방식이 다른 게임들과 비교해 일관성이 없고 부정확함.,타격 메커니즘의 일관성 및 조작감에 대한 사용자 피드백을 정밀 분석하고 개선 방안 검토 필요.,exact_match,None
2,success,144522911,571740,Golf It!,2023-08-18T23:21:37,2023-08-18T00:00:00,0.0,D0-D30,positive,1.000000,0.0,0.500000,mixed,3.0,bug,"[{'category': 'bug', 'sentiment': 'negative', 'evidence': 'I was hitting the ball and not moving.'}, {'category': 'p...",medium,"게임은 매우 재미있으나, 공을 쳐도 움직이지 않는 버그가 발생함.",공 타격 시 캐릭터나 공이 반응하지 않는 현상에 대한 버그 리포트 확인 및 재현 테스트 수행.,partial_match,None
3,success,144527817,571740,Golf It!,2023-08-19T01:06:16,2023-08-18T00:00:00,1.0,D0-D30,positive,53.050000,1.0,0.500000,positive,5.0,positive_praise,"[{'category': 'positive_praise', 'sentiment': 'positive', 'evidence': '$9 가격 대비 풍부한 콘텐츠와 맵 메이커 기능에 만족함'}, {'category...",low,"9달러라는 가격 대비 콘텐츠가 매우 풍부하며, 특히 맵 메이커 기능과 워크숍 지원이 훌륭함.","현재의 풍부한 콘텐츠와 맵 메이커 기능을 유지하고, 커뮤니티 맵 제작을 지속적으로 장려할 것.",exact_match,None
4,success,144536439,571740,Golf It!,2023-08-19T04:08:08,2023-08-18T00:00:00,1.0,D0-D30,positive,131.216667,1.0,0.500000,positive,5.0,positive_praise,"[{'category': 'positive_praise', 'sentiment': 'positive', 'evidence': '개발자의 정성이 느껴지며 재플레이 가치가 높고 즐거운 경험을 제공함'}]",low,"개발자의 정성이 느껴지는 훌륭한 게임이며, 재플레이 가치가 높고 즐거움.","현재의 개발 방향성을 유지하고, 플레이어와의 긍정적인 소통을 지속할 것.",exact_match,None


# 11. 리뷰 단위 결과 후처리 및 저장


In [15]:
def classify_sentiment_relation(row):
    """Steam 라벨과 LLM 감정의 관계 분류"""
    steam_label = row.get("steam_label_text")
    llm_sentiment = row.get("llm_sentiment")

    if steam_label == "positive":
        if llm_sentiment == "positive":
            return "exact_match"
        if llm_sentiment == "mixed":
            return "partial_match"
        if llm_sentiment == "negative":
            return "mismatch"
        return "unclear"

    if steam_label == "negative":
        if llm_sentiment == "negative":
            return "exact_match"
        if llm_sentiment == "mixed":
            return "partial_match"
        if llm_sentiment == "positive":
            return "mismatch"
        return "unclear"

    return "unknown"

# 리뷰 단위 결과 저장 컬럼
REVIEW_RESULT_COLUMNS = [
    # 분석 상태
    "analysis_status",               # LLM 분석 성공/실패 여부를 표시합니다.

    # 원본 리뷰 식별 정보
    "recommendationid",              # 리뷰 고유 ID입니다. Steam 리뷰 1개를 구분하는 식별자입니다.
    "appid",                         # Steam 게임 고유 ID입니다. 어떤 게임의 리뷰인지 구분할 때 사용합니다.
    "game_name",                     # 게임 이름입니다. appid만 보면 알아보기 어려우므로 함께 저장합니다.

    # 리뷰 작성 시점 / 출시 후 구간 정보
    "review_datetime",               # 리뷰 작성 일시입니다. 출시 후 반응 구간을 확인할 때 사용합니다.
    "release_date",                  # 게임 출시일입니다. 리뷰 작성 시점과 비교해 초기/장기 반응을 나눌 때 사용합니다.
    "days_from_release",             # 출시일 기준 리뷰 작성일까지 지난 일수입니다.
    "release_period",                # 출시 후 기간 구간입니다. 예: D0-D7, D8-D30 등입니다.

    # Steam 라벨 / 리뷰 메타 정보
    "steam_label_text",              # Steam 추천 여부를 positive/negative 같은 문자열로 바꾼 값입니다.
    "playtime_at_review_hours",      # 리뷰 작성 시점의 플레이타임입니다. 짧은 플레이 후 부정 리뷰인지 확인할 수 있습니다.
    "votes_up",                      # 해당 리뷰가 받은 '유용함' 투표 수입니다. 리뷰 영향력이나 신뢰도 참고용입니다.
    "weighted_vote_score",           # Steam에서 제공하는 리뷰 가중 점수입니다. 리뷰 노출/신뢰도 참고용입니다.

    # LLM 분석 결과
    "llm_sentiment",                 # LLM이 판단한 리뷰 감정입니다. positive/negative/mixed/neutral 등으로 저장됩니다.
    "sentiment_score",               # LLM이 판단한 감정 점수입니다. 감정 강도를 수치로 비교할 때 사용합니다.
    "llm_primary_issue",                 # LLM이 판단한 리뷰의 대표 이슈입니다. 부정/긍정 반응의 핵심 원인 분류에 사용합니다.
    "llm_issue_tags",                    # 리뷰 안에서 발견된 세부 이슈 태그 목록입니다. 한 리뷰에 여러 문제가 있을 수 있습니다.
    "llm_urgency_candidate",                       # LLM이 리뷰 문맥 기준으로 분류한 시급도 후보입니다. 최종 우선순위가 아니라 보조 참고용입니다.
    "llm_review_summary",                       # LLM이 요약한 리뷰 핵심 내용입니다. 원문을 빠르게 파악하기 위한 요약입니다.
    "llm_suggested_action",              # LLM이 제안한 개선 방향입니다. 패치/운영 방향 제안에 활용합니다.

    # Steam 라벨과 LLM 판단 비교 결과
    "steam_llm_sentiment_relation",            # Steam 추천/비추천 라벨과 LLM 감정 판단이 어느 정도 일치하는지 비교한 값입니다.
]

# 결과 후처리
# df_result: 보고서에서 사용할 성공 분석 결과만 포함한 결과
# df_failed_log: 실패/누락 결과 확인용 메모리 데이터프레임
if len(df_result_raw) > 0:
    df_result_all = df_result_raw.copy()

    # 필수 컬럼 보강
    df_result_all = ensure_columns(
        df_result_all,
        list(dict.fromkeys(REVIEW_RESULT_COLUMNS + ["error_message"]))
    )

    # 타입 안정화
    df_result_all["recommendationid"] = df_result_all["recommendationid"].astype(str)

    for col in ["review_datetime", "release_date"]:
        if col in df_result_all.columns:
            df_result_all[col] = pd.to_datetime(df_result_all[col], errors="coerce")

    # llm_issue_tags는 메모리에서는 list[dict] 형태로 정규화
    df_result_all["llm_issue_tags"] = df_result_all["llm_issue_tags"].apply(parse_issue_tags)

    if {"steam_label_text", "llm_sentiment"}.issubset(df_result_all.columns):
        df_result_all["steam_llm_sentiment_relation"] = df_result_all.apply(classify_sentiment_relation, axis=1)
    else:
        df_result_all["steam_llm_sentiment_relation"] = "unknown"

    if "analysis_status" in df_result_all.columns:
        df_result = df_result_all[df_result_all["analysis_status"] == "success"].copy()
        df_failed_log = df_result_all[df_result_all["analysis_status"] != "success"].copy()
    else:
        df_result = df_result_all.copy()
        df_failed_log = pd.DataFrame(columns=df_result_all.columns)

else:
    df_result_all = pd.DataFrame(columns=list(dict.fromkeys(REVIEW_RESULT_COLUMNS + ["error_message"])))
    df_result = pd.DataFrame(columns=REVIEW_RESULT_COLUMNS)
    df_failed_log = pd.DataFrame(columns=list(dict.fromkeys(REVIEW_RESULT_COLUMNS + ["error_message"])))
    print("분석 결과가 없습니다. 빈 결과 파일을 헤더만 포함해 저장합니다.")


# 저장용 데이터프레임 생성
# 최종 보고서에서 사용하는 핵심 산출물만 저장한다.
df_result_save = ensure_columns(df_result, REVIEW_RESULT_COLUMNS)

# 성공 결과 CSV 저장
# llm_issue_tags는 보고서 코드에서 다시 안전하게 읽을 수 있도록 JSON 문자열로 저장한다.
df_result_save = save_csv_safely(
    df_result_save,
    RESULT_CSV_PATH,
    columns=REVIEW_RESULT_COLUMNS,
    json_cols=["llm_issue_tags"],
)

# 성공 결과 JSON 저장
if SAVE_RESULT_JSON:
    safe_success_results = to_serializable(df_result.to_dict(orient="records"))
    RESULT_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(RESULT_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(safe_success_results, f, ensure_ascii=False, indent=2)
    print("리뷰 단위 LLM 결과 JSON 저장:", RESULT_JSON_PATH)

print("리뷰 단위 LLM 결과 CSV 저장:", RESULT_CSV_PATH)
print("성공 분석 리뷰 수:", len(df_result))
print("실패/누락 리뷰 수:", len(df_failed_log))
print("전체 결과 행 수:", len(df_result_all))


리뷰 단위 LLM 결과 JSON 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_review_analysis_result.json
리뷰 단위 LLM 결과 CSV 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_review_analysis_result.csv
성공 분석 리뷰 수: 4428
실패/누락 리뷰 수: 0
전체 결과 행 수: 4428


# 12. 이슈 태그 펼치기


In [16]:
# 이슈 태그 펼친 결과 저장 컬럼
ISSUE_TAG_FLAT_COLUMNS = [
    "recommendationid",
    "appid",
    "game_name",
    "steam_label_text",
    "llm_sentiment",
    "llm_primary_issue",
    "llm_urgency_candidate",
    "release_period",
    "playtime_at_review_hours",
    "votes_up",
    "weighted_vote_score",
    "llm_issue_category",
    "issue_name_kor",
    "llm_issue_sentiment",
    "llm_issue_evidence",
]


def flatten_issue_tags(df):
    """
    리뷰별 llm_issue_tags 리스트를 이슈 단위 행으로 펼친다.
    """
    flat_rows = []

    if len(df) == 0 or "llm_issue_tags" not in df.columns:
        return pd.DataFrame(columns=ISSUE_TAG_FLAT_COLUMNS)

    for _, row in df.iterrows():
        tags = row.get("llm_issue_tags", [])

        if isinstance(tags, str):
            try:
                tags = json.loads(tags)
            except Exception:
                tags = []

        if not isinstance(tags, list):
            continue

        for tag in tags:
            if not isinstance(tag, dict):
                continue

            category = tag.get("category")
            issue_name_kor = ISSUE_KR_MAP.get(category, category)

            flat_rows.append({
                "recommendationid": row.get("recommendationid"),
                "appid": row.get("appid"),
                "game_name": row.get("game_name"),
                "steam_label_text": row.get("steam_label_text"),
                "llm_sentiment": row.get("llm_sentiment"),
                "llm_primary_issue": row.get("llm_primary_issue"),
                "llm_urgency_candidate": row.get("llm_urgency_candidate"),
                "release_period": row.get("release_period"),
                "playtime_at_review_hours": row.get("playtime_at_review_hours"),
                "votes_up": row.get("votes_up"),
                "weighted_vote_score": row.get("weighted_vote_score"),
                "llm_issue_category": category,
                "issue_name_kor": issue_name_kor,
                "llm_issue_sentiment": tag.get("sentiment"),
                "llm_issue_evidence": tag.get("evidence"),
            })

    return pd.DataFrame(flat_rows, columns=ISSUE_TAG_FLAT_COLUMNS)


df_issue_tags_flat = flatten_issue_tags(df_result)
df_issue_tags_flat.to_csv(ISSUE_TAG_FLAT_PATH, index=False, encoding="utf-8-sig")

print("이슈 태그 펼친 결과 저장:", ISSUE_TAG_FLAT_PATH)
print("이슈 태그 행 수:", len(df_issue_tags_flat))
display(df_issue_tags_flat.head())


이슈 태그 펼친 결과 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_issue_tags_flat.csv
이슈 태그 행 수: 8251


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,llm_primary_issue,llm_urgency_candidate,release_period,playtime_at_review_hours,votes_up,weighted_vote_score,llm_issue_category,issue_name_kor,llm_issue_sentiment,llm_issue_evidence
0,144515690,571740,Golf It!,positive,positive,positive_praise,low,D0-D30,2.916667,6.0,0.563222,positive_praise,긍정 칭찬,positive,I love the game and now that its out of early acces it is even better.
1,144516841,571740,Golf It!,negative,negative,gameplay_loop,high,D0-D30,1.533333,2.0,0.492600,gameplay_loop,게임플레이 루프,negative,The way you hit the ball is too inconsistent in comparison to nearly every other golf game to ever exist.
2,144522911,571740,Golf It!,positive,mixed,bug,medium,D0-D30,1.000000,0.0,0.500000,bug,버그,negative,I was hitting the ball and not moving.
3,144522911,571740,Golf It!,positive,mixed,bug,medium,D0-D30,1.000000,0.0,0.500000,positive_praise,긍정 칭찬,positive,The game is INSANELY fun
4,144527817,571740,Golf It!,positive,positive,positive_praise,low,D0-D30,53.050000,1.0,0.500000,positive_praise,긍정 칭찬,positive,$9 가격 대비 풍부한 콘텐츠와 맵 메이커 기능에 만족함


# 13. 산출물 점검


In [17]:
output_check_targets = {
    "LLM 입력 리뷰 CSV": LLM_INPUT_PATH,
    "필터 로그 CSV": FILTER_LOG_PATH,
    "게임별 샘플 요약 CSV": SAMPLE_SUMMARY_PATH,
    "리뷰별 LLM 분석 결과 CSV": RESULT_CSV_PATH,
    "이슈 태그 펼친 결과 CSV": ISSUE_TAG_FLAT_PATH,
    "LLM 분석 중간 저장 파일": CHECKPOINT_PATH,
}

if SAVE_RESULT_JSON:
    output_check_targets["리뷰별 LLM 분석 결과 JSON"] = RESULT_JSON_PATH

output_check_rows = []

for name, path in output_check_targets.items():
    exists = path.exists()
    rows = None
    columns = None

    if exists and path.suffix.lower() == ".csv":
        try:
            temp_df = pd.read_csv(path)
            rows = len(temp_df)
            columns = len(temp_df.columns)
        except Exception:
            rows = "읽기 실패"
            columns = "읽기 실패"

    output_check_rows.append({
        "산출물": name,
        "exists": exists,
        "rows": rows,
        "columns": columns,
        "path": str(path),
    })

output_check = pd.DataFrame(output_check_rows)
display(output_check)


,산출물,exists,rows,columns,path
0,LLM 입력 리뷰 CSV,True,4429.0,25.0,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_input_reviews.csv
1,필터 로그 CSV,True,9.0,5.0,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_input_filter_log.csv
2,게임별 샘플 요약 CSV,True,163.0,7.0,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_input_sample_summary.csv
3,리뷰별 LLM 분석 결과 CSV,True,4428.0,20.0,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_review_analysis_result.csv
4,이슈 태그 펼친 결과 CSV,True,8251.0,15.0,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_issue_tags_flat.csv
5,LLM 분석 중간 저장 파일,True,NaN,NaN,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_review_analysis_checkpoi...
6,리뷰별 LLM 분석 결과 JSON,True,NaN,NaN,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\llm_review_analysis_result.json


# 14. 출력 테이블 설명


`llm_input_reviews.csv`
- 설명: 출시 전 전체 게임 LLM 리뷰 분류에 실제로 입력된 리뷰 데이터

| 컬럼명                           | 설명                      | 예시 값                                                                      |
| ----------------------------- | ----------------------- | ------------------------------------------------------------------------- |
| `recommendationid`            | Steam 리뷰 고유 ID          | 144515690                                                                 |
| `appid`                       | Steam 게임 고유 ID          | 571740                                                                    |
| `game_name`                   | Steam 게임명               | Golf It!                                                                  |
| `language`                    | 리뷰 작성 언어                | english                                                                   |
| `review_datetime`             | 리뷰 작성 일시                | 2023-08-18 21:03:00                                                       |
| `release_date`                | 게임 출시일                  | 2023-08-18                                                                |
| `days_from_release`           | 출시일 기준 리뷰 작성일까지 지난 일수   | 0                                                                         |
| `release_period`              | 출시 후 리뷰 작성 구간           | D0-D30                                                                    |
| `release_period_detail`       | 출시 후 세부 리뷰 작성 구간        | D0-D7                                                                     |
| `steam_label_text`            | Steam 추천 여부를 문자열로 변환한 값 | positive                                                                  |
| `voted_up`                    | Steam 원본 추천 여부          | True                                                                      |
| `playtime_at_review_hours`    | 리뷰 작성 시점 플레이타임          | 2.9                                                                       |
| `playtime_stage`              | 리뷰 작성 시점 플레이타임 구간       | 1-5h                                                                      |
| `votes_up`                    | 리뷰가 받은 유용함 투표 수         | 6                                                                         |
| `weighted_vote_score`         | Steam 리뷰 가중 점수          | 0.563                                                                     |
| `received_for_free`           | 무료 수령 여부                | False                                                                     |
| `written_during_early_access` | 얼리액세스 기간 작성 여부          | False                                                                     |
| `price`                       | 게임 가격                   | 8.99                                                                      |
| `price_group`                 | 게임 가격 구간                | 5-10                                                                      |
| `genres_text`                 | 게임 장르 목록                | Casual, Indie, Simulation, Sports                                         |
| `categories_text`             | Steam 카테고리 목록           | Single-player, Multi-player                                               |
| `top_steam_tags_text`         | 주요 Steam 태그 목록          | Multiplayer, Mini Golf, Golf                                              |
| `review_text_for_llm`         | LLM에 실제로 전달한 리뷰 본문      | I love the game and now that it is out of early access it is even better. |
| `is_meaningful_review`        | 의미 있는 리뷰 여부             | True                                                                      |
| `meaningless_reason`          | 무의미한 리뷰로 판단된 이유         | meaningful                                                                |



`llm_input_filter_log.csv`
- 설명: LLM 입력 데이터를 만들기 전 필터링 단계별 행 수 변화 기록

| 컬럼명            | 설명              | 예시 값              |
| -------------- | --------------- | ----------------- |
| `step`         | 적용한 필터링 단계명     | 출시 후 D0-D30 리뷰 필터 |
| `before_rows`  | 해당 필터 적용 전 행 수  | 236379            |
| `after_rows`   | 해당 필터 적용 후 행 수  | 12000             |
| `removed_rows` | 해당 필터에서 제거된 행 수 | 224379            |
| `removed_rate` | 제거 비율           | 0.9492            |



`llm_input_sample_summary.csv`
- 설명: LLM 입력으로 샘플링된 게임별 리뷰 수 요약

| 컬럼명                     | 설명                   | 예시 값                |
| ----------------------- | -------------------- | ------------------- |
| `appid`                 | Steam 게임 고유 ID       | 571740              |
| `game_name`             | Steam 게임명            | Golf It!            |
| `sampled_review_count`  | 최종 LLM 입력으로 선택된 리뷰 수 | 48                  |
| `positive_count`        | 샘플 내 Steam 긍정 리뷰 수   | 24                  |
| `negative_count`        | 샘플 내 Steam 부정 리뷰 수   | 24                  |
| `first_review_datetime` | 샘플 내 가장 이른 리뷰 작성 시점  | 2023-08-18 21:03:00 |
| `last_review_datetime`  | 샘플 내 가장 최근 리뷰 작성 시점  | 2023-09-12 12:10:44 |



`llm_review_analysis_result.csv`
- 설명: 리뷰 1개 단위 LLM 감정·이슈 분류 결과

| 컬럼명                            | 설명                                     | 예시 값                                                    |
| ------------------------------ | -------------------------------------- | ------------------------------------------------------- |
| `analysis_status`              | LLM 분석 성공/실패 여부                        | success                                                 |
| `recommendationid`             | Steam 리뷰 고유 ID                         | 144515690                                               |
| `appid`                        | Steam 게임 고유 ID                         | 571740                                                  |
| `game_name`                    | Steam 게임명                              | Golf It!                                                |
| `review_datetime`              | 리뷰 작성 일시                               | 2023-08-18 21:03:00                                     |
| `release_date`                 | 게임 출시일                                 | 2023-08-18                                              |
| `days_from_release`            | 출시일 기준 리뷰 작성일까지 지난 일수                  | 0                                                       |
| `release_period`               | 출시 후 리뷰 작성 구간                          | D0-D30                                                  |
| `steam_label_text`             | Steam 추천/비추천 라벨                        | positive                                                |
| `playtime_at_review_hours`     | 리뷰 작성 시점 플레이타임                         | 2.9                                                     |
| `votes_up`                     | 리뷰가 받은 유용함 투표 수                        | 6                                                       |
| `weighted_vote_score`          | Steam 리뷰 가중 점수                         | 0.563                                                   |
| `llm_sentiment`                | LLM이 판단한 리뷰 전체 감정                      | positive                                                |
| `sentiment_score`              | LLM이 판단한 감정 점수                         | 5                                                       |
| `llm_primary_issue`            | 리뷰의 대표 이슈                              | positive_praise                                         |
| `llm_issue_tags`               | 리뷰 안에서 발견된 세부 이슈 태그 목록                 | [{"category":"positive_praise","sentiment":"positive"}] |
| `llm_urgency_candidate`        | LLM이 리뷰 문맥을 보고 분류한 시급도 후보. 최종 우선순위는 아님 | low                                                     |
| `llm_review_summary`           | LLM이 요약한 리뷰 핵심 내용                      | 정식 출시 이후 게임이 더욱 좋아졌으며 매우 만족함.                           |
| `llm_suggested_action`         | LLM이 리뷰 내용을 바탕으로 정리한 개선 방향 후보          | 현재의 게임 플레이 경험과 정식 출시 버전의 완성도를 유지 및 강화                   |
| `steam_llm_sentiment_relation` | Steam 라벨과 LLM 감정 판단의 관계                | exact_match                                             |



`llm_issue_tags_flat.csv`
- 설명: 리뷰 단위 LLM 결과의 이슈 태그를 1행 1이슈 형태로 펼친 데이터

| 컬럼명                        | 설명                        | 예시 값                                                                   |
| -------------------------- | ------------------------- | ---------------------------------------------------------------------- |
| `recommendationid`         | Steam 리뷰 고유 ID            | 144515690                                                              |
| `appid`                    | Steam 게임 고유 ID            | 571740                                                                 |
| `game_name`                | Steam 게임명                 | Golf It!                                                               |
| `steam_label_text`         | Steam 추천/비추천 라벨           | positive                                                               |
| `llm_sentiment`            | LLM이 판단한 리뷰 전체 감정         | positive                                                               |
| `llm_primary_issue`        | 리뷰의 대표 이슈                 | positive_praise                                                        |
| `llm_urgency_candidate`    | LLM이 리뷰 문맥을 보고 분류한 시급도 후보 | low                                                                    |
| `release_period`           | 출시 후 리뷰 작성 구간             | D0-D30                                                                 |
| `playtime_at_review_hours` | 리뷰 작성 시점 플레이타임            | 2.9                                                                    |
| `votes_up`                 | 리뷰가 받은 유용함 투표 수           | 6                                                                      |
| `weighted_vote_score`      | Steam 리뷰 가중 점수            | 0.563                                                                  |
| `llm_issue_category`       | 세부 이슈 카테고리                | positive_praise                                                        |
| `issue_name_kor`           | 세부 이슈의 한국어 이름             | 긍정 칭찬                                                                  |
| `llm_issue_sentiment`      | LLM이 세부 이슈 단위로 분류한 감정 방향  | positive                                                               |
| `llm_issue_evidence`       | LLM이 해당 태그를 판단한 근거 문장     | I love the game and now that its out of early acces it is even better. |
